#### 그래프 실행
- RunnableConfig
    - recursion_limit: 최대 노드 실행 개수 지정 (순환 로직에 빠지지 않기 위함)
    - thread_id: 그래프 실행 아이디 기록, 추후 추적하기 위한 목적으로 활용됨 (multi-turn 대화할 때, 이 id 별로 따로 대화내용 저장)
    
- 상태(State)로 시작
    - 여기서 "question"에 질문만 입력하고 상태를 첫 번째 노드에 전달
- invoke(상태, config) 전달하여 실행

In [1]:
from graph import build_graph
from langchain_core.runnables import RunnableConfig
from langchain_core.messages import HumanMessage

config = RunnableConfig(
    configurable={
        "thread_id": "research-001",
    }
)

inputs = {"messages": [HumanMessage(content="Domain adaptation in clinical drug")]}
graph = build_graph()
output = graph.invoke(inputs, config)

print(output["messages"][-1].content)

/scratch/hpc201a03/.conda/envs/gapag/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


LangSmith 추적을 시작합니다.
[프로젝트명]
GAPAGO
ROUTE: NEED_USER_CLARIFICATION

query_proposal: "domain adaptation methods applied to clinical drug development or drug response prediction using real-world clinical data"

missing_slots:
- specific task (e.g., drug response prediction, drug repurposing, adverse drug reaction detection, clinical trial outcome prediction)
- data modality (e.g., EHR, genomics, transcriptomics, imaging, claims data)
- adaptation setting (e.g., cross-hospital, cross-population, preclinical-to-clinical, trial-to-real-world)
- method family (e.g., deep learning, representation learning, causal inference, transfer learning)
- time scope (e.g., last 5 years, no restriction)
- evaluation focus (e.g., predictive performance, generalization, bias/fairness, clinical utility)

clarify_questions:
- What specific clinical drug-related task are you interested in (prediction, discovery, safety, efficacy, repurposing)?
- What type of data should be considered (EHR, omics, clinical tri

In [2]:
# 그래프 상태 스냅샷 생성
snapshot = graph.get_state(config)

# 다음 스냅샷 상태 접근
snapshot.next

('human_clarify',)

In [3]:
graph.update_state(
    config,
    {
        "query_approved": True,
        "ask_human": False,
        "messages": [HumanMessage(content="APPROVE")],
    },
)

for event in graph.stream(None, config, stream_mode="values"):
    if "messages" in event:
        print("==>", event["messages"][-1].content)

==> APPROVE
==> {"refined_query": "", "keywords": [], "negative_keywords": [], "expanded_terms": [], "arxiv_query_candidates": [], "web_query_candidates": [], "scienceon_query_candidates": [], "notes": ["No refined_query or keywords were provided. Retrieval expansions require a topic or seed terms. Please supply a refined_query or initial keywords."]}
==> ```json
{
  "selected_tools": ["arxiv_api_call_tool"],
  "tool_rationale": "The query concerns domain adaptation in a clinical drug context, which is best addressed through academic ML/biomedical literature. arXiv provides sufficient coverage for machine learning, transfer learning, and drug response prediction studies. Web search was not required at this stage.",
  "papers": [
    {
      "paper_id": "arxiv:2502.04034v2",
      "title": "Fourier Asymmetric Attention on Domain Generalization for Pan-Cancer Drug Response Prediction",
      "year": 2025,
      "url": "https://arxiv.org/abs/2502.04034v2",
      "abstract": "Proposes a do

In [ ]:
output2 = graph.invoke(None, config)
print(output2["messages"][-1].content)